# WET-007: Paired vs Unpaired Zernike Estimation

Owner: **John Franklin Crenshaw** <br>
Last Verified to Run: **2024-10-28** <br>
Software Version:
  - `ts_wep`: **12.6.0** (on branch `tickets/DM-47188`)
  - `lsst_distrib`: **w_2024_43**

## Test Description

This notebook addresses [SITCOM-1149](https://rubinobs.atlassian.net/browse/SITCOM-1149) by comparing performance of paired vs unpaired Zernike estimation.

Data for this analyis was created by running the following two commands:

TIE:
```sh
pipetask run -j 4 -b /repo/embargo -i LSSTComCam/defaults -o u/crenshaw/WET-007_tie_unpaired_20241027_3 -p /home/c/crenshaw/notebooks/comcam_commissioning/prep_for_WET-007_unpaired/pipeline_WET-007_tie_unpaired.yaml -d "instrument='LSSTComCam' and exposure.observation_type='cwfs' and visit in (2024102700016,2024102700017)" --rebase --register-dataset-types
```

Danish:
```sh
pipetask run -j 4 -b /repo/embargo -i LSSTComCam/defaults -o u/crenshaw/WET-007_danish_unpaired_20241027_3 -p /home/c/crenshaw/notebooks/comcam_commissioning/prep_for_WET-007_unpaired/pipeline_WET-007_danish_unpaired.yaml -d "instrument='LSSTComCam' and exposure.observation_type='cwfs' and visit in (2024102700016,2024102700017)" --rebase --register-dataset-types
```

where the contents of `pipeline_WET-007_tie_unpaired.yaml` are
```yaml
description: Pipeline for WET-007 using TIE, with unpaired donuts
instrument: lsst.obs.lsst.LsstComCam
tasks:
  generateDonutDirectDetectTask:
    class: lsst.ts.wep.task.generateDonutDirectDetectTask.GenerateDonutDirectDetectTask
    config:
      donutSelector.useCustomMagLimit: True
      donutSelector.sourceLimit: 5
  cutOutDonutsUnpairedTask:
    class: lsst.ts.wep.task.cutOutDonutsUnpairedTask.CutOutDonutsUnpairedTask
    config:
      donutStampSize: 160
      initialCutoutPadding: 40
  calcZernikesUnpairedTask:
    class: lsst.ts.wep.task.calcZernikesUnpairedTask.CalcZernikesUnpairedTask
    config:
      estimateZernikes.maxNollIndex: 28
      estimateZernikes.saveHistory: False
      estimateZernikes.maskKwargs: {'doMaskBlends': False}
  isr:
    class: lsst.ip.isr.IsrTaskLSST
    config:
      # Although we don't have to apply the amp offset corrections, we do want
      # to compute them for analyzeAmpOffsetMetadata to report on as metrics.
      doAmpOffset: true
      ampOffset.doApplyAmpOffset: false
      # Turn off slow steps in ISR
      doBrighterFatter: false
      # Mask saturated pixels,
      # but turn off quadratic crosstalk because it's currently broken
      doSaturation: True
      crosstalk.doQuadraticCrosstalkCorrection: False
```

and the contents of `pipeline_WET-007_danish_unpaired.yaml` are
```yaml
description: Pipeline for WET-007 using Danish, with unpaired donuts
instrument: lsst.obs.lsst.LsstComCam
tasks:
  generateDonutDirectDetectTask:
    class: lsst.ts.wep.task.generateDonutDirectDetectTask.GenerateDonutDirectDetectTask
    config:
      donutSelector.useCustomMagLimit: True
      donutSelector.sourceLimit: 5
  cutOutDonutsUnpairedTask:
    class: lsst.ts.wep.task.cutOutDonutsUnpairedTask.CutOutDonutsUnpairedTask
    config:
      donutStampSize: 160
      initialCutoutPadding: 40
  calcZernikesUnpairedTask:
    class: lsst.ts.wep.task.calcZernikesUnpairedTask.CalcZernikesUnpairedTask
    config:
      estimateZernikes.maxNollIndex: 28
      estimateZernikes.saveHistory: False
      python: |
        from lsst.ts.wep.task import EstimateZernikesDanishTask
        config.estimateZernikes.retarget(EstimateZernikesDanishTask)
  isr:
    class: lsst.ip.isr.IsrTaskLSST
    config:
      # Although we don't have to apply the amp offset corrections, we do want
      # to compute them for analyzeAmpOffsetMetadata to report on as metrics.
      doAmpOffset: true
      ampOffset.doApplyAmpOffset: false
      # Turn off slow steps in ISR
      doBrighterFatter: false
      # Mask saturated pixels,
      # but turn off quadratic crosstalk because it's currently broken
      doSaturation: True
      crosstalk.doQuadraticCrosstalkCorrection: False
```

# Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lsst.daf.butler import Butler

In [ ]:
# Change this path to appropriate butler when on-sky images arrive
path_to_aos_butler = '/repo/embargo'
butler = Butler(path_to_aos_butler)

tie_collection = "u/crenshaw/WET-007_tie_unpaired_20241027_3"
danish_collection = "u/crenshaw/WET-007_danish_unpaired_20241027_3"

## Plot donut stamps

First we will plot the donut stamps to see what they look like.

Above each stamp I print the field angle of the donut so we can identify whether the TIE and Danish donut pairs are the same.

In [ ]:
def plot_stamps(dataId, collections, axes, title=""):
    # Load the stamps
    stamps = butler.get("donutStamps", dataId=dataId, collections=collections)
    
    # Filter by stamp selection
    quality = butler.get("donutQualityTable", dataId=dataId, collections=collections)
    selected = quality["FINAL_SELECT"]
    stamps = [stamp for stamp, sel in zip(stamps, selected) if sel]

    # Determine if the Zernikes from each stamp were used
    usedList = butler.get("zernikes", dataId=dataId, collections=collections)[1:]["used"]

    # Determine the defocal type
    defocalType = quality["DEFOCAL_TYPE"][0]

    for ax, stamp, used in zip(axes, stamps, usedList):
        # Plot the stamp
        ax.imshow(stamp.wep_im.image, origin="lower")

        # Print field angle above image
        angle = stamp.wep_im.fieldAngle
        ax.set_title(f"({angle[0]:.2f}, {angle[1]:.2f})", fontsize=8)

        # If the Zernikes weren't used, plot a red X
        if not used:
            npix = stamp.wep_im.image.shape[0] - 1
            ax.plot([0, npix], [0, npix], c="r", lw=1)
            ax.plot([0, npix], [npix, 0], c="r", lw=1)
            ax.plot([0, npix], [0, npix], c="r", lw=1)
            ax.plot([0, npix], [npix, 0], c="r", lw=1)

    # Hide axes that weren't filled
    for ax in axes[len(usedList):]:
        ax.set_axis_off()

    # Hide pixel ticks
    for ax in axes:
        ax.set(xticks=[], yticks=[])

    # Set the title for the row
    axes[0].set_ylabel(f"{title}\n{defocalType}")

In [ ]:
# Get IDs for all the data
tie_ids = list(butler.registry.queryDataIds(('exposure', 'visit', 'detector'), collections=tie_collection, datasets='donutStamps'))
dan_ids = list(butler.registry.queryDataIds(('exposure', 'visit', 'detector'), collections=danish_collection, datasets='donutStamps'))

# Sort into intra/extra pairs
# (even though they weren't paired for estimation)
tie_ids = np.array(sorted(set(tie_ids))).reshape(-1, 2)
dan_ids = np.array(sorted(set(dan_ids))).reshape(-1, 2)

# Create figure for all the stamps
nrows = 4 * len(tie_ids)
fig, axes = plt.subplots(nrows, 5, figsize=(8, 1.75 * nrows), constrained_layout=True)

# Loop through all the IDs and plot stamps
for i, (tieID, danID) in enumerate(zip(tie_ids, dan_ids)):
    det = tieID[0]["detector"]
    plot_stamps(tieID[0], tie_collection, axes[4*i], title=f"Detector {det}\nTIE")
    plot_stamps(tieID[1], tie_collection, axes[4*i+1], title=f"Detector {det}\nTIE")
    plot_stamps(danID[0], tie_collection, axes[4*i+2], title=f"Detector {det}\nDanish")
    plot_stamps(danID[1], tie_collection, axes[4*i+3], title=f"Detector {det}\nDanish")

## Load Zernike Estimates

Below we plot the average of the unpaired, single-side-of-focus Zernike estimates

In [ ]:
# Load zernike estimates
tie_ids = list(butler.registry.queryDataIds(('exposure', 'visit', 'detector'), collections=tie_collection, datasets='zernikes'))
tie_zks = []
for data_id in tie_ids:
    table = butler.get('zernikes', dataId=data_id, collections=tie_collection)
    tie_zks.append(np.array([table[table["label"] == "average"][f"Z{i}"][0].value for i in range(4, 29)]))
tie_zks = np.array(tie_zks)

dan_ids = list(butler.registry.queryDataIds(('exposure', 'visit', 'detector'), collections=danish_collection, datasets='zernikes'))
dan_zks = []
for data_id in tie_ids:
    table = butler.get('zernikes', dataId=data_id, collections=danish_collection)
    dan_zks.append(np.array([table[table["label"] == "average"][f"Z{i}"][0].value for i in range(4, 29)]))
dan_zks = np.array(dan_zks)

In [ ]:
fig, ax = plt.subplots(1, 1, dpi=150)

ax.errorbar(np.arange(4, 29), np.mean(tie_zks, axis=0), yerr=np.std(tie_zks, axis=0), ls="--", c="C2", label="TIE", capsize=4)
ax.errorbar(np.arange(4, 29), np.mean(dan_zks, axis=0), yerr=np.std(dan_zks, axis=0), ls="--", c="cornflowerblue", alpha=1, label="Danish", capsize=4)
ax.scatter([4, 5, 6, 7, 8], [-1573.3, 819.4, 1290.6, -1920.3, 671.3], c="r", marker="x", label="Josh", zorder=10, s=70)

ax.legend()

ax.set(
    xlabel="Noll index",
    ylabel="Amplitude [nm]",
    xticks=np.arange(4, 29, 2),
    xlim=(3.5, 28.5),
)

# inset Axes....
x1, x2, y1, y2 = 3.5, 8.75, -2650, +2350  # subregion of the original image
axins = ax.inset_axes(
    [0.5, 0.075, 0.4, 0.35],
    xlim=(x1, x2), ylim=(y1, y2), xticks=np.arange(4, 9), yticks=[-2e3, -1e3, 0, 1e3, 2e3], ylabel="nm")

axins.errorbar(np.arange(4, 29), np.mean(tie_zks, axis=0), yerr=np.std(tie_zks, axis=0), ls="--", c="C2", label="TIE", capsize=4)
axins.errorbar(np.arange(4, 29), np.mean(dan_zks, axis=0), yerr=np.std(dan_zks, axis=0), ls="--", c="cornflowerblue", alpha=1, label="Danish", capsize=4)
axins.scatter([4, 5, 6, 7, 8], [-1573.3, 819.4, 1290.6, -1920.3, 671.3], c="r", marker="x", label="Josh", zorder=10, s=70)


ax.indicate_inset_zoom(axins, edgecolor="gray")

plt.show()

The 4 points labeled "Josh" were created manually by Josh Meyers, who used Danish to jointly fit a single intra- & extra-focal donut. Points above are means of the combined Zernikes from each of the 9 detectors. The uncertainties are the standard deviations of these combined Zernikes.

The TIE and Danish mostly agree within their purported uncertainties, and look consistent with the paired Zernikes!
This is great news!

# Comparing paired vs unpaired scatter for individual detectors

These plots look at the scatter within "used" Zernike estimates for individual detectors, and compare them for paired vs unpaired estimates.

In [ ]:
def get_zk_scatter(dataId, collections):
    table = butler.get('zernikes', dataId=dataId, collections=collections)
    table = table[(table["label"] != "average") & table["used"]]
    zks = table[[f"Z{i}" for i in range(4, 29)]].to_pandas().values
    return zks.std(axis=0)

def get_defocal_type(dataId, collections):
    qual_table = butler.get("donutQualityTable", dataId, collections=collections)
    defocal_type = qual_table["DEFOCAL_TYPE"][0]
    return defocal_type

In [ ]:
tie_unpaired_collection = tie_collection
tie_paired_collection = "u/crenshaw/WET-007_tie_20241027_3"
tie_unpaired = list(butler.registry.queryDataIds(('exposure', 'visit', 'detector'), collections=tie_unpaired_collection, datasets='zernikes'))
tie_paired = list(butler.registry.queryDataIds(('exposure', 'visit', 'detector'), collections=tie_paired_collection, datasets='zernikes'))
tie_unpaired = np.array(sorted(set(tie_unpaired))).reshape(-1, 2)
tie_paired = sorted(set(tie_paired))

dan_unpaired_collection = danish_collection
dan_paired_collection = "u/crenshaw/WET-007_danish_20241027_3"
dan_unpaired = list(butler.registry.queryDataIds(('exposure', 'visit', 'detector'), collections=dan_unpaired_collection, datasets='zernikes'))
dan_paired = list(butler.registry.queryDataIds(('exposure', 'visit', 'detector'), collections=dan_paired_collection, datasets='zernikes'))
dan_unpaired = np.array(sorted(set(dan_unpaired))).reshape(-1, 2)
dan_paired = sorted(set(dan_paired))

for tu, tp, du, dp in zip(tie_unpaired, tie_paired, dan_unpaired, dan_paired):
    fig, (ax1, ax2) = plt.subplots(1, 2, dpi=120, constrained_layout=True, figsize=(6, 2))

    # TIE unpaired, Intra
    scatter = get_zk_scatter(tu[0], tie_unpaired_collection)
    defocal_type = get_defocal_type(tu[0], tie_unpaired_collection)
    ax1.plot(np.arange(4, len(scatter)+4), scatter, label=defocal_type, c="C1", ls="--")

    # TIE unpaired, Extra
    scatter = get_zk_scatter(tu[1], tie_unpaired_collection)
    defocal_type = get_defocal_type(tu[1], tie_unpaired_collection)
    ax1.plot(np.arange(4, len(scatter)+4), scatter, label=defocal_type, c="C2", ls=":")

    # TIE paired
    scatter = get_zk_scatter(tp, tie_paired_collection)
    ax1.plot(np.arange(4, len(scatter)+4), scatter, label="paired", c="C0", ls="-", lw=1)
    
    ax1.legend()
    ax1.set(title="TIE")

    # Danish unpaired, Intra
    scatter = get_zk_scatter(du[0], dan_unpaired_collection)
    defocal_type = get_defocal_type(du[0], dan_unpaired_collection)
    ax2.plot(np.arange(4, len(scatter)+4), scatter, label=defocal_type, c="C1", ls="--")

    # Danish unpaired, Extra
    scatter = get_zk_scatter(du[1], dan_unpaired_collection)
    defocal_type = get_defocal_type(du[1], dan_unpaired_collection)
    ax2.plot(np.arange(4, len(scatter)+4), scatter, label=defocal_type, c="C2", ls=":")

    # Danish paired
    scatter = get_zk_scatter(dp, dan_paired_collection)
    ax2.plot(np.arange(4, len(scatter)+4), scatter, label="paired", c="C0", ls="-", lw=1)
    
    ax2.legend()
    ax2.set(title="Danish")

    fig.suptitle(f"Detector {tp['detector']}")

There is no clear trend here for intra vs extra vs paired variances.

# Discussion

Everything looks good in this analysis.
This makes it look like we can start doing single-side-of-focus analyses, especially if we use Danish.
However, I do think we should do more complex offline analyses using single-side-of-focus to validate the approach.
E.g., I would like to duplicate the sensitivity matrix analysis using intra-only and/or extra-only Zernikes.